# ABSA Pipeline: Aspect-Based Sentiment Analysis
## Dataset: IMDb Reviews in Spanish (50,000 filas)

**Arquitectura:**
- **SpaCy** (`es_core_news_md`): Extracción sintáctica de pares (Aspecto, Adjetivo)
- **BERT Fine-tuning**: Ajustar `edumunozsala/beto_sentiment_analysis_es` al dominio IMDb
- **BERT Inferencia**: Clasificación de sentimiento por fragmento con modelo fine-tuneado
- **Output**: Parquet estructurado + pesos del modelo fine-tuneado

**Flujo de ejecución en Colab:**
1. **Celda 1** — Instalar dependencias
2. **Celda 2** — Descargar dataset desde Kaggle
3. **Celda 3** — Extracción de aspectos con SpaCy
4. **Celda 4a** — Fine-tuning de BETO
5. **Celda 4b** — Inferencia BERT con modelo fine-tuneado
6. **Celda 5** — Consolidación y exportación final

**Outputs:**
- `/content/output/aspectos_sentimientos_final.parquet` — Dataset final (listo para Streamlit)
- `/content/output/metricas_por_aspecto.parquet` — Métricas agregadas por aspecto
- `/content/output/beto-imdb-finetuned/` — Pesos del modelo fine-tuneado

---

### Celda 1: Instalación e Importación de Librerías

In [ ]:
# Celda 1: Instalación de Dependencias
# Ejecutar UNA SOLA VEZ al inicio de la sesión de Colab

!pip install -q pandas pyarrow spacy transformers torch tqdm kaggle datasets accelerate evaluate scikit-learn
!python -m spacy download es_core_news_md -q

print("✅ Instalación completada. Reiniciar runtime si se solicita.")

In [ ]:
# Celda 1: Importaciones

import os
import sys
import gc
import json
import logging
import zipfile
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Generator, Optional

import pandas as pd
import numpy as np
import torch
import spacy
from spacy.tokens import Doc
from tqdm.notebook import tqdm
from transformers import pipeline

# Verificar disponibilidad de GPU
print(f"🔥 PyTorch versión: {torch.__version__}")
print(f"🚀 GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"📊 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ Advertencia: No se detectó GPU. El procesamiento será más lento.")

### Celda 2: Configuración del Entorno y Descarga del Dataset

**Objetivo:** Descargar el dataset desde Kaggle y configurar rutas locales.

**Estrategia de datos:**
- Descargar dataset usando API de Kaggle (requiere `kaggle.json` subido a Colab)
- Usar rutas locales de Colab (`/content/`) para evitar dependencias de Drive
- Cargar CSV con tipos optimizados (`category` para sentimiento)
- **Muestreo configurable**: `SAMPLE_SIZE` permite probar con subsets (ej: 1000) antes de procesar las 50k filas

In [ ]:
# Celda 2: Configuración de Rutas y Logging

# Rutas locales de Colab (sin dependencias de Drive)
BASE_PATH = Path("/content")
DATA_RAW = BASE_PATH / "data" / "raw"
DATA_PROCESSED = BASE_PATH / "data" / "processed"
DATA_MODELS = BASE_PATH / "data" / "models"
OUTPUT_PATH = BASE_PATH / "output"
LOGS_PATH = BASE_PATH / "logs"

# Crear directorios
for path in [DATA_RAW, DATA_PROCESSED, DATA_MODELS, OUTPUT_PATH, LOGS_PATH]:
    path.mkdir(parents=True, exist_ok=True)
    print(f"📁 {path}")

# Configurar logging
log_file = LOGS_PATH / f"absa_pipeline_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler(sys.stdout)
    ]
)

logger = logging.getLogger(__name__)
logger.info("🚀 Pipeline ABSA iniciado")

print("\n✅ Entorno configurado correctamente.")

In [ ]:
# Celda 2: Descarga del Dataset desde Kaggle

KAGGLE_DATASET = "lucamla/imdb-reviews-in-spanish"
CSV_FILENAME = "IMDB Dataset Spanish.csv"

csv_path = DATA_RAW / CSV_FILENAME

if not csv_path.exists():
    logger.info("📥 Descargando dataset desde Kaggle...")
    
    # Configurar API de Kaggle
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    
    # Intentar copiar kaggle.json desde varias ubicaciones
    kaggle_json_paths = [
        "/content/kaggle.json",
        os.path.expanduser("~/kaggle.json"),
    ]
    
    kaggle_configured = False
    for kaggle_path in kaggle_json_paths:
        if os.path.exists(kaggle_path):
            !cp {kaggle_path} ~/.kaggle/kaggle.json
            !chmod 600 ~/.kaggle/kaggle.json
            kaggle_configured = True
            logger.info(f"✅ Kaggle configurado desde: {kaggle_path}")
            break
    
    if not kaggle_configured:
        logger.warning("⚠️ kaggle.json no encontrado. Sube el archivo a Colab o configura manualmente.")
        logger.info("💡 Alternativa: sube el CSV directamente a /content/data/raw/")
    else:
        # Descargar dataset
        !kaggle datasets download -d {KAGGLE_DATASET} -p {DATA_RAW}
        
        # Descomprimir si viene en zip
        zip_files = list(DATA_RAW.glob("*.zip"))
        if zip_files:
            with zipfile.ZipFile(zip_files[0], 'r') as zip_ref:
                zip_ref.extractall(DATA_RAW)
            logger.info(f"📦 Dataset descomprimido: {zip_files[0].name}")
else:
    logger.info("📂 Dataset ya existe en cache. Omitiendo descarga.")

In [ ]:
# Celda 2: Carga del Dataset con Tipos Optimizados

# Configuración de muestreo (ajustar según necesidad)
# Configuración de muestreo (None = usar todo el dataset)SAMPLE_SIZE = None  # None para usar todo, 1000 para pruebas rápidas
logger.info(f"🔄 Cargando dataset con muestreo: {SAMPLE_SIZE} filas")

# Encontrar archivo CSV
csv_files = list(DATA_RAW.rglob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"❌ No se encontró CSV en {DATA_RAW}")

# Seleccionar el CSV más grande
csv_path = max(csv_files, key=lambda p: p.stat().st_size)
logger.info(f"📄 Archivo detectado: {csv_path.name}")

# Cargar con tipos optimizados
dtypes = {
    'line number': 'int32',
    'review_en': 'string',
    'review_es': 'string',
    'sentiment': 'category',
    'sentimiento': 'category'
}

# Solo usar columnas que existan
usecols = [c for c in dtypes.keys() if c in pd.read_csv(csv_path, nrows=0).columns]
logger.info(f"📋 Columnas detectadas: {usecols}")

df = pd.read_csv(csv_path, dtype={k: v for k, v in dtypes.items() if k in usecols}, usecols=usecols)

# Aplicar muestreo estratificado por sentimiento
if SAMPLE_SIZE is not None and SAMPLE_SIZE < len(df):    if 'sentiment' in df.columns:
        df = df.groupby('sentiment', group_keys=False).apply(
            lambda x: x.sample(int(np.rint(SAMPLE_SIZE * len(x) / len(df))), random_state=42)
        ).sample(frac=1, random_state=42).reset_index(drop=True)
    else:
        df = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
    logger.info(f"✂️ Muestreo aplicado: {len(df)} filas")

# Validar columnas requeridas
assert 'review_es' in df.columns, "❌ Falta columna 'review_es'"

# Eliminar filas con texto vacío
df = df.dropna(subset=['review_es'])
df = df[df['review_es'].str.len() > 10]

# Normalizar ID de reseña
if 'line number' not in df.columns:
    df['line number'] = df.index
    logger.info("🆔 Usando índice como review_id")

logger.info(f"✅ Dataset cargado: {len(df)} filas | Memoria: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB")
if 'sentiment' in df.columns:
    logger.info(f"📊 Distribución de sentimientos: {dict(df['sentiment'].value_counts())}")

df.head(3)

### Celda 3: Módulo de Extracción Sintáctica (SpaCy)

**Objetivo:** Extraer pares `(Aspecto, Adjetivo)` de cada reseña usando SpaCy con procesamiento por lotes.

**Optimizaciones aplicadas:**
- **`nlp.pipe(batch_size=256)`**: Procesamiento vectorizado
- **Deshabilitar componentes**: `disable=['ner', 'textcat', 'entity_linker']` para ahorrar CPU
- **Lematización**: Normaliza aspectos para evitar duplicados
- **Manejo de errores**: Try-except por reseña

**Lógica de extracción:**
1. Para cada oración, identificar sustantivos (`NOUN`, `PROPN`)
2. Buscar hijos con dependencia `amod` (adjetivo modificador)
3. Extraer lema del sustantivo (aspecto) y forma del adjetivo
4. Guardar fragmento original para contexto del BERT

In [ ]:
# Celda 3: Funciones de Extracción SpaCy (inline)

def cargar_modelo_spacy(modelo: str = "es_core_news_md") -> spacy.Language:
    """Carga el modelo de SpaCy deshabilitando componentes no necesarios."""
    logger.info(f"Cargando modelo SpaCy: {modelo}")
    nlp = spacy.load(modelo, disable=["ner", "textcat", "entity_linker"])
    return nlp


def extraer_aspectos_adjetivos(doc: Doc) -> List[Dict[str, str]]:
    """Extrae pares (sustantivo aspecto, adjetivo modificador) de un Doc de SpaCy."""
    resultados = []
    
    for sent in doc.sents:
        for token in sent:
            if token.pos_ in ("NOUN", "PROPN"):
                adjetivos = [
                    child for child in token.children 
                    if child.dep_ == "amod" and child.pos_ == "ADJ"
                ]
                
                for adj in adjetivos:
                    fragmento = doc[token.i:adj.i + 1].text
                    resultados.append({
                        "aspecto_lematizado": token.lemma_.lower(),
                        "adjetivo": adj.text.lower(),
                        "adjetivo_lematizado": adj.lemma_.lower(),
                        "fragmento": fragmento,
                        "dep": adj.dep_,
                        "pos_aspecto": token.pos_,
                        "pos_adjetivo": adj.pos_
                    })
    
    return resultados


def extraer_fragmentos_para_bert(
    textos: List[str],
    ids: List[int],
    nlp: spacy.Language,
    batch_size: int = 256,
    incluir_texto_completo: bool = False
) -> List[Dict]:
    """Extrae fragmentos listos para enviar al modelo BERT de sentimiento."""
    resultados = []
    
    for review_id, doc in zip(ids, nlp.pipe(textos, batch_size=batch_size)):
        try:
            if not doc or len(doc.text.strip()) == 0:
                continue
            
            aspectos = extraer_aspectos_adjetivos(doc)
            
            for aspecto in aspectos:
                registro = {
                    "review_id": review_id,
                    "aspecto": aspecto["aspecto_lematizado"],
                    "adjetivo": aspecto["adjetivo"],
                    "adjetivo_lematizado": aspecto["adjetivo_lematizado"],
                    "fragmento": aspecto["fragmento"],
                }
                if incluir_texto_completo:
                    registro["texto_completo"] = doc.text
                resultados.append(registro)
        
        except Exception as e:
            logger.error(f"Error procesando review_id={review_id}: {str(e)}")
            continue
    
    return resultados


print("✅ Funciones SpaCy definidas.")

In [ ]:
# Celda 3: Configuración del Modelo SpaCy

SPACY_MODEL = "es_core_news_md"
BATCH_SIZE_SPACY = 256

logger.info(f"🔧 Cargando modelo SpaCy: {SPACY_MODEL}")

nlp = cargar_modelo_spacy(SPACY_MODEL)

if 'sentencizer' not in nlp.pipe_names:
    nlp.add_pipe('sentencizer')

logger.info(f"✅ Modelo cargado. Componentes activos: {nlp.pipe_names}")

In [ ]:
# Celda 3: Ejecución de la Extracción de Aspectos

gc.collect()

textos = df['review_es'].tolist()
ids = df['line number'].tolist()

logger.info(f"🎯 Iniciando extracción de aspectos para {len(textos)} reseñas...")

aspectos_extraidos = extraer_fragmentos_para_bert(
    textos=textos,
    ids=ids,
    nlp=nlp,
    batch_size=BATCH_SIZE_SPACY,
    incluir_texto_completo=True
)

df_aspectos = pd.DataFrame(aspectos_extraidos)

logger.info(f"📊 Total aspectos extraídos: {len(df_aspectos)}")
logger.info(f"📁 Memoria del DataFrame: {df_aspectos.memory_usage(deep=True).sum() / 1e6:.2f} MB")

print("\n🎭 Top 10 Aspectos más frecuentes:")
print(df_aspectos['aspecto'].value_counts().head(10))

print("\n🌈 Top 10 Adjetivos más frecuentes:")
print(df_aspectos['adjetivo'].value_counts().head(10))

print("\n💡 Ejemplos de extracción:")
df_aspectos[['review_id', 'aspecto', 'adjetivo', 'fragmento']].head(10)

In [ ]:
# Celda 3: Checkpoint Intermedio

checkpoint_spacy = DATA_PROCESSED / "checkpoint_aspectos_spacy.parquet"
df_aspectos.to_parquet(checkpoint_spacy, index=False, compression='snappy')

logger.info(f"💾 Checkpoint guardado: {checkpoint_spacy}")
logger.info(f"📦 Tamaño del archivo: {checkpoint_spacy.stat().st_size / 1e6:.2f} MB")

del textos
del ids
del aspectos_extraidos
gc.collect()

print("✅ Checkpoint guardado. Listo para Celda 4a (Fine-tuning).")

### Celda 4a: Fine-tuning del Modelo BERT

**Objetivo:** Ajustar `edumunozsala/beto_sentiment_analysis_es` al dominio específico de reseñas de IMDb en español.

**¿Por qué BETO?**
- BETO es un modelo BERT-base entrenado en español con Whole Word Masking
- `edumunozsala/beto_sentiment_analysis_es` ya tiene head de clasificación de sentimiento pre-entrenado
- Fine-tuning adapta el modelo al vocabulario cinematográfico de IMDb

**Estrategia de contexto completo (nueva):**
- El modelo se entrena con pares `(fragmento + contexto de reseña)` en lugar de solo el fragmento
- Formato: `"El villano es malo [SEP] Contexto: Esta película es brillante..."`
- Esto enseña al modelo que un adjetivo como "malo" puede ser positivo si describe a un personaje bien construido en una reseña positiva
- Reduce el error de clasificar aspectos basándose solo en la polaridad del adjetivo

**Configuración:**
- **Modelo base:** `edumunozsala/beto_sentiment_analysis_es`
- **Datos:** Pares aspecto-adjetivo con contexto de reseña + labels binarios `positive`/`negative`
- **Split:** 80% train / 10% validation / 10% test (estratificado)
- **Framework:** `transformers.Trainer` con `accelerate` para GPU
- **Épocas:** Máximo 5 (con early stopping paciencia=5)
- **Batch size:** 16 (por VRAM limitada en T4)
- **Max length:** 256 tokens (para capturar fragmento + contexto)
- **Salida:** `/content/output/beto-imdb-finetuned/`

**Nuevas métricas:** Accuracy, F1, Precision, Recall en validation y test

**Output:**
- Pesos del modelo fine-tuneado guardados localmente
- `training_metrics.json` — Métricas detalladas de entrenamiento, validación y test
- `label_config.json` — Mapeo de labels y métricas
- El modelo se usará en Celda 4b para inferencia


In [ ]:
# Celda 4a: Preparación del Dataset para Fine-tuning

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback, DataCollatorWithPadding
import evaluate
from sklearn.model_selection import train_test_split
import numpy as np

# ───────────────────────────────────────────────
# CONFIGURACIÓN DEL FINE-TUNING
# ───────────────────────────────────────────────
BERT_BASE_MODEL = "edumunozsala/beto_sentiment_analysis_es"
FINETUNED_MODEL_PATH = OUTPUT_PATH / "beto-imdb-finetuned"
FINETUNING_EPOCHS = 5
FINETUNING_BATCH_SIZE = 16
FINETUNING_LEARNING_RATE = 2e-5
FINETUNING_WARMUP_RATIO = 0.1
MAX_LENGTH = 256
CONTEXT_MAX_LEN = 300  # Caracteres del contexto a incluir

logger.info(f"🔧 Preparando fine-tuning de {BERT_BASE_MODEL}")
logger.info(f"📐 Split: 80% train / 10% validation / 10% test")

# ───────────────────────────────────────────────
# PASO 1: Cargar aspectos extraídos con texto completo
# ───────────────────────────────────────────────
checkpoint_spacy = DATA_PROCESSED / "checkpoint_aspectos_spacy.parquet"
if not checkpoint_spacy.exists():
    raise FileNotFoundError("❌ Ejecutar Celda 3 primero para extraer aspectos.")

df_aspectos_ft = pd.read_parquet(checkpoint_spacy)
logger.info(f"📂 Aspectos cargados: {len(df_aspectos_ft)} filas")

# Verificar que tengamos texto_completo
if "texto_completo" not in df_aspectos_ft.columns:
    raise ValueError("❌ El checkpoint no tiene columna 'texto_completo'. Re-ejecutar Celda 3 con incluir_texto_completo=True.")

# ───────────────────────────────────────────────
# PASO 2: Construir ejemplos de entrenamiento CON CONTEXTO
# Formato: "fragmento [SEP] Contexto: texto_completo_truncado"
# Esto enseña al modelo que el sentimiento de un fragmento
# depende del contexto global de la reseña.
# ───────────────────────────────────────────────
def construir_input_con_contexto(fragmento, texto_completo):
    contexto = texto_completo[:CONTEXT_MAX_LEN] if texto_completo else ""
    return f"{fragmento} [SEP] Contexto: {contexto}"

df_aspectos_ft["text"] = df_aspectos_ft.apply(
    lambda row: construir_input_con_contexto(row["fragmento"], row["texto_completo"]),
    axis=1
)

# ───────────────────────────────────────────────
# PASO 3: Obtener labels del dataset original
# ───────────────────────────────────────────────
if "sentiment" not in df.columns:
    raise ValueError("❌ El dataset no tiene columna 'sentiment' para fine-tuning")

# Merge con el sentiment original de cada review
df_labels = df[["line number", "sentiment"]].copy()
df_labels.columns = ["review_id", "sentiment"]
df_aspectos_ft = df_aspectos_ft.merge(df_labels, on="review_id", how="left")

# Mapear labels a IDs numéricos (binario: positive/negative)
label_map = {"positive": 1, "negative": 0}
label_map_inv = {1: "positive", 0: "negative"}
df_aspectos_ft["labels"] = df_aspectos_ft["sentiment"].map(label_map)
df_aspectos_ft = df_aspectos_ft.dropna(subset=["labels"])
df_aspectos_ft["labels"] = df_aspectos_ft["labels"].astype(int)

logger.info(f"📊 Ejemplos con label válido: {len(df_aspectos_ft)}")
logger.info(f"🏷️  Distribución: {dict(df_aspectos_ft['labels'].value_counts().sort_index())}")

# ───────────────────────────────────────────────
# PASO 4: Split estratificado 80/10/10
# ───────────────────────────────────────────────
# Primero: separar 80% train y 20% temp
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df_aspectos_ft["text"].values,
    df_aspectos_ft["labels"].values,
    test_size=0.20,
    random_state=42,
    stratify=df_aspectos_ft["labels"].values
)

# Luego: separar temp en 50% val / 50% test (10% cada uno del total)
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels,
    test_size=0.50,
    random_state=42,
    stratify=temp_labels
)

# Crear datasets de Hugging Face
train_dataset = Dataset.from_dict({"text": train_texts, "labels": train_labels})
val_dataset = Dataset.from_dict({"text": val_texts, "labels": val_labels})
test_dataset = Dataset.from_dict({"text": test_texts, "labels": test_labels})

dataset = {
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
}

logger.info(f"📊 Split final: {len(dataset['train'])} train / {len(dataset['validation'])} val / {len(dataset['test'])} test")
logger.info(f"🏷️  Distribución train: {dict(pd.Series(dataset['train']['labels']).value_counts().sort_index())}")

# Guardar configuración de split
split_config = {
    "train_size": len(dataset["train"]),
    "val_size": len(dataset["validation"]),
    "test_size": len(dataset["test"]),
    "label_map": label_map,
    "label_map_inv": label_map_inv,
    "context_max_len": CONTEXT_MAX_LEN,
    "base_model": BERT_BASE_MODEL
}

print(f"✅ Dataset preparado: {len(dataset['train'])} ejemplos de entrenamiento (con contexto)")
print(f"✅ Validación: {len(dataset['validation'])} | Test: {len(dataset['test'])}")


In [ ]:
# Celda 4a: Tokenización y Carga del Modelo Base

logger.info(f"🔧 Cargando tokenizer y modelo base: {BERT_BASE_MODEL}")

# Cargar tokenizer de BETO
tokenizer = AutoTokenizer.from_pretrained(BERT_BASE_MODEL)

# Función de tokenización (sin padding para eficiencia)
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

# Tokenizar los 3 splits
tokenized_datasets = {}
for split in ["train", "validation", "test"]:
    tokenized_datasets[split] = dataset[split].map(tokenize_function, batched=True)
    # Eliminar columnas de texto originales para ahorrar memoria
    tokenized_datasets[split] = tokenized_datasets[split].remove_columns(["text"])
    tokenized_datasets[split].set_format("torch")
    logger.info(f"✅ Tokenizado {split}: {len(tokenized_datasets[split])} ejemplos")

# Data collator con padding dinámico (más eficiente que max_length)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Cargar modelo base (2 clases: positive/negative)
model = AutoModelForSequenceClassification.from_pretrained(
    BERT_BASE_MODEL,
    num_labels=2,
    ignore_mismatched_sizes=True
)

logger.info(f"✅ Modelo cargado: {model.config.model_type} | Clases: {model.num_labels}")
logger.info(f"✅ Tokenizer vocab size: {len(tokenizer)}")

# Liberar memoria antes del entrenamiento
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("✅ Modelo, tokenizer y data collator listos para entrenamiento")


In [ ]:
# Celda 4a: Configuración del Trainer y Entrenamiento

# ───────────────────────────────────────────────
# Métricas de evaluación extendidas
# ───────────────────────────────────────────────
metric_accuracy = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")
metric_precision = evaluate.load("precision")
metric_recall = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    accuracy = metric_accuracy.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = metric_f1.compute(predictions=predictions, references=labels, average="weighted")["f1"]
    precision = metric_precision.compute(predictions=predictions, references=labels, average="weighted")["precision"]
    recall = metric_recall.compute(predictions=predictions, references=labels, average="weighted")["recall"]
    
    return {
        "accuracy": accuracy,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

# ───────────────────────────────────────────────
# Argumentos de entrenamiento
# ───────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=str(DATA_MODELS / "checkpoints"),
    num_train_epochs=FINETUNING_EPOCHS,
    per_device_train_batch_size=FINETUNING_BATCH_SIZE,
    per_device_eval_batch_size=FINETUNING_BATCH_SIZE,
    learning_rate=FINETUNING_LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=FINETUNING_WARMUP_RATIO,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    report_to="none",
    seed=42,
)

# ───────────────────────────────────────────────
# Early Stopping con paciencia=5
# ───────────────────────────────────────────────
early_stopping = EarlyStoppingCallback(
    early_stopping_patience=5,
    early_stopping_threshold=0.001
)

# ───────────────────────────────────────────────
# Inicializar Trainer
# ───────────────────────────────────────────────
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    callbacks=[early_stopping]
)

logger.info(f"🚀 Iniciando fine-tuning: {FINETUNING_EPOCHS} épocas máximas, batch={FINETUNING_BATCH_SIZE}")
logger.info(f"🛑 Early stopping: paciencia=5, umbral=0.001")

# Entrenar
train_result = trainer.train()

# Guardar métricas de entrenamiento
logger.info(f"📊 Métricas finales de entrenamiento: {train_result.metrics}")

print("\n" + "="*60)
print("🎉 FINE-TUNING COMPLETADO")
print("="*60)
print(f"📊 Loss final: {train_result.metrics['train_loss']:.4f}")
print(f"📈 Tiempo de entrenamiento: {train_result.metrics['train_runtime']:.1f}s")
print(f"🔄 Épocas ejecutadas: {train_result.metrics.get('epoch', 'N/A')}")
print("="*60)


In [ ]:
# Celda 4a: Evaluación y Guardado del Modelo Fine-tuneado

# ───────────────────────────────────────────────
# PASO 1: Evaluar en validation set (durante entrenamiento)
# ───────────────────────────────────────────────
eval_val_results = trainer.evaluate(tokenized_datasets["validation"])
logger.info(f"📊 Resultados validación: {eval_val_results}")

print("\n📊 MÉTRICAS DE VALIDACIÓN:")
print(f"  Accuracy: {eval_val_results['eval_accuracy']:.4f}")
print(f"  F1 Score: {eval_val_results['eval_f1']:.4f}")
print(f"  Precision: {eval_val_results['eval_precision']:.4f}")
print(f"  Recall: {eval_val_results['eval_recall']:.4f}")
print(f"  Loss: {eval_val_results['eval_loss']:.4f}")

# ───────────────────────────────────────────────
# PASO 2: Evaluar en TEST set (nunca visto durante entrenamiento)
# ───────────────────────────────────────────────
eval_test_results = trainer.evaluate(tokenized_datasets["test"])
logger.info(f"📊 Resultados test: {eval_test_results}")

print("\n📊 MÉTRICAS DE TEST (hold-out):")
print(f"  Accuracy: {eval_test_results['eval_accuracy']:.4f}")
print(f"  F1 Score: {eval_test_results['eval_f1']:.4f}")
print(f"  Precision: {eval_test_results['eval_precision']:.4f}")
print(f"  Recall: {eval_test_results['eval_recall']:.4f}")
print(f"  Loss: {eval_test_results['eval_loss']:.4f}")

# ───────────────────────────────────────────────
# PASO 3: Guardar modelo fine-tuneado
# ───────────────────────────────────────────────
logger.info(f"💾 Guardando modelo fine-tuneado en: {FINETUNED_MODEL_PATH}")

trainer.save_model(str(FINETUNED_MODEL_PATH))
tokenizer.save_pretrained(str(FINETUNED_MODEL_PATH))

# ───────────────────────────────────────────────
# PASO 4: Guardar métricas detalladas en JSON
# ───────────────────────────────────────────────
metrics_all = {
    "model": str(FINETUNED_MODEL_PATH.name),
    "base_model": BERT_BASE_MODEL,
    "timestamp": datetime.now().isoformat(),
    "training": train_result.metrics,
    "validation": {k: v for k, v in eval_val_results.items() if not isinstance(v, (np.ndarray, torch.Tensor))},
    "test": {k: v for k, v in eval_test_results.items() if not isinstance(v, (np.ndarray, torch.Tensor))},
    "dataset": {
        "train_size": len(dataset["train"]),
        "val_size": len(dataset["validation"]),
        "test_size": len(dataset["test"]),
        "context_max_len": CONTEXT_MAX_LEN,
        "max_length": MAX_LENGTH,
    },
    "hyperparameters": {
        "epochs": FINETUNING_EPOCHS,
        "batch_size": FINETUNING_BATCH_SIZE,
        "learning_rate": FINETUNING_LEARNING_RATE,
        "warmup_ratio": FINETUNING_WARMUP_RATIO,
        "weight_decay": 0.01,
        "early_stopping_patience": 5,
    },
    "label_map": label_map,
    "label_map_inv": label_map_inv,
    "environment": {
        "pytorch_version": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    }
}

# Convertir valores numpy/float32 a float nativo para JSON
def convert_to_native(obj):
    if isinstance(obj, dict):
        return {k: convert_to_native(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_native(v) for v in obj]
    elif isinstance(obj, (np.floating, np.integer)):
        return float(obj)
    return obj

metrics_path = FINETUNED_MODEL_PATH / "training_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(convert_to_native(metrics_all), f, indent=2, ensure_ascii=False)

logger.info(f"📊 Métricas guardadas: {metrics_path}")

# ───────────────────────────────────────────────
# PASO 5: Guardar configuración de labels (compatibilidad con pipeline)
# ───────────────────────────────────────────────
config_path = FINETUNED_MODEL_PATH / "label_config.json"
with open(config_path, "w") as f:
    json.dump({
        "id2label": label_map_inv,
        "label2id": label_map,
        "base_model": BERT_BASE_MODEL,
        "training_samples": len(dataset["train"]),
        "eval_accuracy": float(eval_val_results["eval_accuracy"]),
        "eval_f1": float(eval_val_results["eval_f1"]),
        "test_accuracy": float(eval_test_results["eval_accuracy"]),
        "test_f1": float(eval_test_results["eval_f1"])
    }, f, indent=2)

logger.info(f"✅ Modelo guardado: {FINETUNED_MODEL_PATH}")
model_size_mb = sum(f.stat().st_size for f in FINETUNED_MODEL_PATH.rglob("*") if f.is_file()) / 1e6
logger.info(f"📦 Tamaño total: {model_size_mb:.2f} MB")

# ───────────────────────────────────────────────
# LIMPIEZA DE MEMORIA
# ───────────────────────────────────────────────
del trainer
del model
del tokenizer
del tokenized_datasets
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"\n✅ Modelo fine-tuneado guardado en: {FINETUNED_MODEL_PATH}")
print(f"✅ Métricas detalladas guardadas en: {metrics_path}")
print("✅ Listo para Celda 4b (Inferencia con modelo fine-tuneado)")


### Celda 4b: Inferencia con Modelo Fine-tuneado

**Objetivo:** Clasificar el sentimiento de cada fragmento extraído usando el modelo fine-tuneado en Celda 4a.

**Optimizaciones aplicadas:**
- **`pipeline(device=0)`**: Fuerza uso de GPU (CUDA)
- **`batch_size=64` o `128`**: Procesamiento por lotes para saturar la GPU
- **`truncation=True, max_length=256`**: Evita exceder límite de tokens (aumentado de 128 a 256)
- **`torch.cuda.empty_cache()`**: Libera VRAM entre lotes grandes
- **Manejo de errores**: Si un batch falla, se salta y continúa

**Mapeo de labels (BETO):**
- `LABEL_1` → positive
- `LABEL_0` → negative

**Nota:** Si el modelo fine-tuneado no existe, se usará `edumunozsala/beto_sentiment_analysis_es` como fallback.


In [ ]:
# Celda 4b: Funciones de Análisis de Sentimiento (inline)

def configurar_pipeline_sentimiento(
    modelo: str = "edumunozsala/beto_sentiment_analysis_es",
    device: int = 0,
    batch_size: int = 64,
    max_length: int = 128
):
    """Configura el pipeline de Hugging Face para análisis de sentimiento."""
    if device >= 0 and not torch.cuda.is_available():
        logger.warning("CUDA no disponible. Forzando CPU.")
        device = -1
    
    logger.info(f"Cargando modelo: {modelo} | Dispositivo: {'GPU' if device >= 0 else 'CPU'}")
    
    classifier = pipeline(
        "sentiment-analysis",
        model=modelo,
        tokenizer=modelo,
        device=device,
        truncation=True,
        max_length=max_length,
        batch_size=batch_size
    )
    
    return classifier


def analizar_sentimiento_lote(
    fragmentos: List[str],
    classifier,
    batch_size: int = 64
) -> List[Dict]:
    """Clasifica el sentimiento de una lista de fragmentos de texto."""
    resultados = []
    total_fragmentos = len(fragmentos)
    logger.info(f"Iniciando inferencia: {total_fragmentos} fragmentos, batch={batch_size}")
    
    for i in range(0, total_fragmentos, batch_size):
        lote = fragmentos[i:i + batch_size]
        
        try:
            predicciones = classifier(lote)
            
            for pred in predicciones:
                resultados.append({
                    "sentimiento": pred["label"],
                    "confianza": round(pred["score"], 4),
                    "label_original": pred["label"]
                })
        
        except Exception as e:
            logger.error(f"Error en batch {i//batch_size}: {str(e)}")
            for _ in lote:
                resultados.append({
                    "sentimiento": "ERROR",
                    "confianza": 0.0,
                    "label_original": "ERROR"
                })
        
        if i % (batch_size * 10) == 0 and torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    logger.info(f"Inferencia completada: {len(resultados)} resultados")
    return resultados


def obtener_mapeo_sentimientos(ruta_modelo: Optional[str] = None) -> Dict[str, str]:
    """Obtiene el mapeo de labels según el tipo de modelo."""
    if ruta_modelo:
        config_path = Path(ruta_modelo) / "label_config.json"
        if config_path.exists():
            with open(config_path, 'r') as f:
                config = json.load(f)
            if 'id2label' in config:
                return {f"LABEL_{k}": v for k, v in config['id2label'].items()}
    
    return {
        "POS": "positive",
        "NEG": "negative",
        "NEU": "neutral"
    }


print("✅ Funciones de análisis de sentimiento definidas.")

In [ ]:
# Celda 4b: Configuración del Pipeline BERT con Modelo Fine-tuneado

# Configuración del modelo BERT para sentimiento
FINETUNED_MODEL_PATH = OUTPUT_PATH / "beto-imdb-finetuned"
BATCH_SIZE_BERT = 64
MAX_LENGTH = 256

# Determinar qué modelo usar
if FINETUNED_MODEL_PATH.exists() and (FINETUNED_MODEL_PATH / "config.json").exists():
    BERT_MODEL = str(FINETUNED_MODEL_PATH)
    logger.info(f"✅ Usando modelo fine-tuneado: {BERT_MODEL}")
    
    # Cargar configuración de labels
    with open(FINETUNED_MODEL_PATH / "label_config.json", "r") as f:
        label_config = json.load(f)
    sentiment_map = {f"LABEL_{k}": v for k, v in label_config["id2label"].items()}
    print(f"📊 Métricas validación: Accuracy={label_config['eval_accuracy']:.4f}, F1={label_config['eval_f1']:.4f}")
    print(f"📊 Métricas test: Accuracy={label_config['test_accuracy']:.4f}, F1={label_config['test_f1']:.4f}")
else:
    BERT_MODEL = "edumunozsala/beto_sentiment_analysis_es"
    logger.warning(f"⚠️ Modelo fine-tuneado no encontrado. Usando modelo base: {BERT_MODEL}")
    # BETO usa LABEL_0 (NEG) y LABEL_1 (POS)
    sentiment_map = {
        "LABEL_0": "negative",
        "LABEL_1": "positive"
    }

# Verificar GPU antes de cargar modelo
if torch.cuda.is_available():
    device = 0
    print(f"🚀 Usando GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM disponible: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    device = -1
    print("⚠️ GPU no disponible. Usando CPU (será más lento).")

logger.info(f"🔧 Cargando modelo BERT: {BERT_MODEL}")

# Configurar pipeline de inferencia
classifier = configurar_pipeline_sentimiento(
    modelo=BERT_MODEL,
    device=device,
    batch_size=BATCH_SIZE_BERT,
    max_length=MAX_LENGTH
)

print("✅ Pipeline BERT configurado correctamente.")


In [ ]:
# Celda 4b: Ejecución de Inferencia BERT

# Cargar checkpoint de SpaCy si no está en memoria
if 'df_aspectos' not in globals():
    checkpoint_spacy = DATA_PROCESSED / "checkpoint_aspectos_spacy.parquet"
    if checkpoint_spacy.exists():
        df_aspectos = pd.read_parquet(checkpoint_spacy)
        logger.info(f"📂 Checkpoint SpaCy cargado: {len(df_aspectos)} filas")
    else:
        raise FileNotFoundError("❌ No se encontró checkpoint de SpaCy. Ejecutar Celda 3 primero.")

# Preparar fragmentos para BERT
fragmentos = df_aspectos['fragmento'].tolist()

logger.info(f"🎯 Enviando {len(fragmentos)} fragmentos al modelo BERT...")

# Ejecutar inferencia
resultados_sentimiento = analizar_sentimiento_lote(
    fragmentos=fragmentos,
    classifier=classifier,
    batch_size=BATCH_SIZE_BERT
)

# Crear DataFrame de resultados BERT
df_sentimientos = pd.DataFrame(resultados_sentimiento)
df_sentimientos['sentimiento'] = df_sentimientos['label_original'].map(sentiment_map).fillna('unknown')

# Unir con aspectos
df_aspectos['sentimiento_bert'] = df_sentimientos['sentimiento']
df_aspectos['confianza'] = df_sentimientos['confianza']
df_aspectos['label_original'] = df_sentimientos['label_original']

# Mostrar distribución de sentimientos
print("\n📊 Distribución de Sentimientos por Aspecto:")
print(df_aspectos['sentimiento_bert'].value_counts())

print("\n💡 Ejemplos con sentimiento:")
print(df_aspectos[['aspecto', 'adjetivo', 'fragmento', 'sentimiento_bert', 'confianza']].head(10))

# Liberar memoria
del fragmentos
del resultados_sentimiento
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

logger.info("✅ Celda 4b completada. Memoria liberada.")

In [ ]:
# Celda 4b: Checkpoint BERT

checkpoint_bert = DATA_PROCESSED / "checkpoint_aspectos_sentimientos.parquet"
df_aspectos.to_parquet(checkpoint_bert, index=False, compression='snappy')

logger.info(f"💾 Checkpoint BERT guardado: {checkpoint_bert}")
logger.info(f"📦 Tamaño del archivo: {checkpoint_bert.stat().st_size / 1e6:.2f} MB")

print("✅ Checkpoint BERT guardado. Listo para Celda 5 (Consolidación Final).")

### Celda 5: Orquestación Final y Guardado

**Objetivo:** Consolidar todos los resultados, calcular métricas agregadas por aspecto y exportar el archivo final optimizado para Streamlit.

**Procesos realizados:**
1. **Merge con datos originales**: Unir `sentiment` original del dataset con predicciones BERT
2. **Agregación por aspecto**: Calcular métricas desglosadas (conteo, confianza promedio, sentimiento dominante)
3. **Normalización**: Eliminar aspectos poco frecuentes (ruido) y estandarizar formato
4. **Exportación final**: Parquet comprimido con todas las columnas necesarias para el dashboard

**Estructura del output final (`aspectos_sentimientos_final.parquet`):**
- `review_id`: ID original de la reseña
- `aspecto`: Sustantivo lematizado (aspecto extraído)
- `adjetivo`: Adjetivo tal cual aparece en el texto
- `fragmento`: Contexto completo del par aspecto-adjetivo
- `sentimiento_bert`: Etiqueta predicha (positive/negative)
- `confianza`: Score de confianza del modelo BERT
- `sentiment_original`: Polaridad original del dataset (pos/neg)
- `metricas_aspecto`: Conteo, confianza promedio y sentimiento dominante por aspecto

In [ ]:
# Celda 5: Consolidación Final y Exportación

# Cargar checkpoint si no está en memoria
if 'df_aspectos' not in globals():
    checkpoint_bert = DATA_PROCESSED / "checkpoint_aspectos_sentimientos.parquet"
    if checkpoint_bert.exists():
        df_aspectos = pd.read_parquet(checkpoint_bert)
        logger.info(f"📂 Checkpoint BERT cargado: {len(df_aspectos)} filas")
    else:
        raise FileNotFoundError("❌ No se encontró checkpoint BERT. Ejecutar Celda 4b primero.")

# ───────────────────────────────────────────────
# PASO 1: Merge con datos originales del dataset
# ───────────────────────────────────────────────

logger.info("🔄 Iniciando consolidación final...")

if 'df' in globals():
    cols_origen = ['line number']
    if 'sentiment' in df.columns:
        cols_origen.append('sentiment')
    if 'sentimiento' in df.columns:
        cols_origen.append('sentimiento')
    
    df_original = df[cols_origen].copy()
    df_original.columns = ['review_id'] + [f'sentiment_{c}' for c in cols_origen[1:]]
    
    df_final = df_aspectos.merge(
        df_original,
        on='review_id',
        how='left'
    )
else:
    logger.warning("⚠️ DataFrame original no disponible. Continuando sin merge.")
    df_final = df_aspectos.copy()
    df_final['sentiment_original'] = 'unknown'

# ───────────────────────────────────────────────
# PASO 2: Agregación de métricas por aspecto
# ───────────────────────────────────────────────

logger.info("📊 Calculando métricas agregadas por aspecto...")

metricas_aspecto = df_final.groupby('aspecto').agg(
    total_apariciones=('aspecto', 'size'),
    confianza_promedio=('confianza', 'mean'),
    sentimiento_dominante=('sentimiento_bert', lambda x: x.mode().iloc[0] if not x.mode().empty else 'neutral'),
    pct_positivo=('sentimiento_bert', lambda x: (x == 'positive').mean() * 100),
    pct_negativo=('sentimiento_bert', lambda x: (x == 'negative').mean() * 100)
).reset_index()

# Filtrar aspectos con poca frecuencia (ruido)
MIN_FRECUENCIA = 5
aspectos_validos = metricas_aspecto[metricas_aspecto['total_apariciones'] >= MIN_FRECUENCIA]['aspecto'].tolist()

df_final = df_final[df_final['aspecto'].isin(aspectos_validos)].copy()
metricas_aspecto = metricas_aspecto[metricas_aspecto['total_apariciones'] >= MIN_FRECUENCIA].copy()

logger.info(f"🎯 Aspectos filtrados (>= {MIN_FRECUENCIA} apariciones): {len(aspectos_validos)}")

# ───────────────────────────────────────────────
# PASO 3: Optimizar tipos de datos para exportación
# ───────────────────────────────────────────────

df_final['sentimiento_bert'] = df_final['sentimiento_bert'].astype('category')
df_final['aspecto'] = df_final['aspecto'].astype('category')
df_final['confianza'] = df_final['confianza'].astype('float32')

# ───────────────────────────────────────────────
# PASO 4: Exportar resultado final a Parquet
# ───────────────────────────────────────────────

OUTPUT_FILE = OUTPUT_PATH / "aspectos_sentimientos_final.parquet"
METRICAS_FILE = OUTPUT_PATH / "metricas_por_aspecto.parquet"

df_final.to_parquet(OUTPUT_FILE, index=False, compression='snappy')
logger.info(f"💾 Archivo final guardado: {OUTPUT_FILE}")
logger.info(f"📦 Tamaño: {OUTPUT_FILE.stat().st_size / 1e6:.2f} MB")

metricas_aspecto.to_parquet(METRICAS_FILE, index=False, compression='snappy')
logger.info(f"💾 Métricas por aspecto guardadas: {METRICAS_FILE}")

# ───────────────────────────────────────────────
# RESUMEN FINAL
# ───────────────────────────────────────────────

print("\n" + "="*60)
print("🎉 PIPELINE ABSA COMPLETADO CON ÉXITO")
print("="*60)
print(f"📊 Total de reseñas procesadas: {df_final['review_id'].nunique()}")
print(f"🎭 Total de pares aspecto-adjetivo: {len(df_final)}")
print(f"🏷️  Aspectos únicos encontrados: {df_final['aspecto'].nunique()}")
print(f"📁 Archivo final: {OUTPUT_FILE.name}")
print(f"📈 Métricas por aspecto: {METRICAS_FILE.name}")
print(f"🧠 Modelo fine-tuneado: {FINETUNED_MODEL_PATH}")
print("="*60)

print("\n📊 Distribución de Sentimientos (BERT):")
print(df_final['sentimiento_bert'].value_counts())

print("\n🎭 Top 10 Aspectos más mencionados:")
print(df_final['aspecto'].value_counts().head(10))

print("\n🌈 Top 10 Adjetivos más frecuentes:")
print(df_final['adjetivo'].value_counts().head(10))

print("\n💡 Vista previa del output final:")
print(df_final[['review_id', 'aspecto', 'adjetivo', 'sentimiento_bert', 'confianza']].head())

# ───────────────────────────────────────────────
# LIMPIEZA FINAL DE MEMORIA
# ───────────────────────────────────────────────

logger.info("🧹 Limpiando memoria...")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

logger.info("✅ Pipeline finalizado. Todo listo para Streamlit! 🚀")

### 🎉 Pipeline Completado

**Archivos generados en `/content/output/`:**

1. `aspectos_sentimientos_final.parquet` — Dataset completo con aspectos y sentimientos (listo para Streamlit)
2. `metricas_por_aspecto.parquet` — Métricas agregadas por aspecto
3. `beto-imdb-finetuned/` — Pesos del modelo fine-tuneado (~440MB)
   - `config.json` — Configuración del modelo
   - `model.safetensors` — Pesos del modelo
   - `tokenizer.json` — Tokenizer
   - `label_config.json` — Mapeo de labels y métricas de evaluación
   - `training_metrics.json` — Métricas detalladas (train/val/test)

**Checkpoints intermedios en `/content/data/processed/`:**
4. `checkpoint_aspectos_spacy.parquet` — Checkpoint intermedio de SpaCy (ahora incluye texto_completo)
5. `checkpoint_aspectos_sentimientos.parquet` — Checkpoint intermedio de BERT

**Próximos pasos:**
- Descargar los archivos Parquet y el modelo desde Colab
- Construir dashboard de Streamlit consumiendo `aspectos_sentimientos_final.parquet`
- Usar el modelo fine-tuneado para inferencia en producción
- Visualizar: distribución de aspectos, nubes de adjetivos, evolución de sentimiento por review
